In [1]:
# Check GPU availability
!nvidia-smi


Fri Jan 30 23:06:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile hello_world.cu
#include <cuda_runtime.h>
#include <iostream>

__global__ void hello_world(void) {
    printf("block idx: %d thread idx: %d\n", blockIdx.x, threadIdx.x);
    if (threadIdx.x == 0) {
        printf("GPU: Hello world!\n");
    }
}

int main(int argc, char **argv) {
    printf("CPU: Hello world\n"); 
    hello_world<<<1, 10>>>();
    cudaDeviceSynchronize();
    if (cudaGetLastError() != cudaSuccess) {
        std::cerr << "CUDA Error: " << cudaGetErrorString(cudaGetLastError()) << std::endl;
        return 1;
    } else {
        std::cout << "GPU: Hello world finished!" << std::endl;
    }
    std::cout << "CPU: Hello world finished!" << std::endl;
    return 0;
}

Overwriting hello_world.cu


In [6]:
!nvcc hello_world.cu -o hello_world \
-arch=sm_75 \
-O3 \
--ptxas-options=-v \
-lineinfo

ptxas info    : 49 bytes gmem, 24 bytes cmem[4]
ptxas info    : Compiling entry function '_Z11hello_worldv' for 'sm_75'
ptxas info    : Function properties for _Z11hello_worldv
    8 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 24 registers, 8 bytes cumulative stack size, 352 bytes cmem[0]


In [7]:
!./hello_world

CPU: Hello world
block idx: 0 thread idx: 0
block idx: 0 thread idx: 1
block idx: 0 thread idx: 2
block idx: 0 thread idx: 3
block idx: 0 thread idx: 4
block idx: 0 thread idx: 5
block idx: 0 thread idx: 6
block idx: 0 thread idx: 7
block idx: 0 thread idx: 8
block idx: 0 thread idx: 9
GPU: Hello world!
GPU: Hello world finished!
CPU: Hello world finished!


## Other Useful Compilation Flags